# 02 - Preprocessing

Goal: create a clean, balanced binary dataset for the first GAT experiment.

Decision for the first model:

- `Benign` -> `0`
- every other label -> `1`

We keep source and destination IP columns because the next step will build a graph: IP addresses are nodes, traffic flows are edges.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocess_binary import (  # noqa: E402
    CATEGORICAL_COLUMNS,
    IP_COLUMNS,
    NUMERIC_COLUMNS,
    TARGET_COLUMN,
    make_binary_sample,
    save_splits,
    write_metadata,
)

DATA_PATH = PROJECT_ROOT / "data" / "iot23_combined_new.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Data exists:", DATA_PATH.exists())

## Configuration

The first GAT experiment should be small enough to run on a laptop. Later we can increase `ROWS_PER_CLASS`.

In [ ]:
ROWS_PER_CLASS = 50_000
CHUNK_SIZE = 500_000
RANDOM_STATE = 42

print("Rows per class:", ROWS_PER_CLASS)
print("Expected total rows:", ROWS_PER_CLASS * 2)

## Create the binary sample

This cell scans the full CSV in chunks, cleans the values, and keeps a balanced random sample.

In [ ]:
sample_df, summary = make_binary_sample(
    data_path=DATA_PATH,
    rows_per_class=ROWS_PER_CLASS,
    chunk_size=CHUNK_SIZE,
    random_state=RANDOM_STATE,
)

summary

In [ ]:
display(sample_df.head())
print("Shape:", sample_df.shape)
print("Binary label counts:")
display(sample_df[TARGET_COLUMN].value_counts().sort_index().to_frame("count"))
print("Raw label counts:")
display(sample_df["label"].value_counts().to_frame("count"))

## Save train, validation, and test files

We split after sampling and keep the binary labels balanced in each split.

In [ ]:
paths = save_splits(
    sample=sample_df,
    output_dir=OUTPUT_DIR,
    random_state=RANDOM_STATE,
)

args = Namespace(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    rows_per_class=ROWS_PER_CLASS,
    chunk_size=CHUNK_SIZE,
    random_state=RANDOM_STATE,
)
write_metadata(OUTPUT_DIR, summary, paths, args)

paths

In [ ]:
train_df = pd.read_csv(OUTPUT_DIR / "train.csv")
val_df = pd.read_csv(OUTPUT_DIR / "val.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test.csv")

split_summary = pd.DataFrame({
    "rows": [len(train_df), len(val_df), len(test_df)],
    "benign": [
        (train_df[TARGET_COLUMN] == 0).sum(),
        (val_df[TARGET_COLUMN] == 0).sum(),
        (test_df[TARGET_COLUMN] == 0).sum(),
    ],
    "malicious": [
        (train_df[TARGET_COLUMN] == 1).sum(),
        (val_df[TARGET_COLUMN] == 1).sum(),
        (test_df[TARGET_COLUMN] == 1).sum(),
    ],
}, index=["train", "val", "test"])

split_summary

## Columns for the next step

The next notebook will turn these flows into a PyTorch Geometric graph.

In [ ]:
print("IP columns:", IP_COLUMNS)
print("Numeric columns:", NUMERIC_COLUMNS)
print("Categorical columns:", CATEGORICAL_COLUMNS)
print("Target column:", TARGET_COLUMN)